In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


CPU threads set to: 32
CUDA device: NVIDIA GeForce RTX 3090



# LLaVA-1.5 7B zero-shot baseline — Kvasir-VQA x1

Single-GPU zero-shot evaluation on the x1 validation + test splits. Reuses the x1 answer normalization/top-K mapping so we can compare with BLIP/BLIP-2 style baselines. No training.


In [2]:

# !pip install sentencepiece

import os
import json
import random
import re
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from sklearn.metrics import classification_report, f1_score
from transformers import AutoProcessor, LlavaForConditionalGeneration

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

try:
    import importlib.metadata as _im
    _im.version("bitsandbytes")
    BNB_AVAILABLE = True
except Exception:
    BNB_AVAILABLE = False
    BitsAndBytesConfig = None
    print("bitsandbytes not found; disabling 4-bit/8-bit quantization.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


2026-01-31 13:10:48.276926: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-31 13:10:48.276955: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-31 13:10:48.277850: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-31 13:10:48.282565: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-31 13:10:48.944326: W tensorflow/compiler/tf2

bitsandbytes not found; disabling 4-bit/8-bit quantization.


In [3]:

# Paths & run config

def find_kvasir_x1_root() -> Path:
    import os
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "Kvasir_VQA_x1" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "Kvasir_VQA_x1" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate Kvasir_VQA_x1 dataset root. \n"
        "Run this notebook from within the Kvasir_VQA_x1 folder, \n"
        "or set KVASIR_VQA_X1_ROOT."
    )

DATA_ROOT = find_kvasir_x1_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"

RUN_NAME = "03_vlm_modern_baseline_zeroshot"
MODEL_NAME = os.environ.get("VLM_MODEL_NAME", "llava-hf/llava-1.5-7b-hf")
SPLITS = ["validation", "test"]
SAMPLE_N: Optional[int] = None  # set an int to subsample each split for smoke tests
TOP_K = 200  # follow x1 top-K convention

BATCH_SIZE = 2  # adjust if OOM
MAX_NEW_TOKENS = 16
USE_4BIT = True   # preferred for single 24/32GB GPUs
USE_8BIT = False  # fallback if 4-bit unavailable

OUT_DIR = DATA_ROOT / "2_modeling" / RUN_NAME / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Output dir:", OUT_DIR)
print("Model:", MODEL_NAME)


Data root: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Output dir: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/03_vlm_modern_baseline_zeroshot/out
Model: llava-hf/llava-1.5-7b-hf


In [4]:

# Load metadata & build top-K answers from train split

def basic_norm(text: str) -> str:
    t = str(text).lower().strip()
    t = re.sub(r"[^a-z0-9\s\-]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

meta_all = pd.read_csv(META_CSV)
images_base = DATA_ROOT / "0_dataset_prep"
meta_all["image_path"] = meta_all["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

# Basic cleaning
meta_all = meta_all.dropna(subset=["question", "answer", "image_path"]).reset_index(drop=True)
meta_all = meta_all[meta_all["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

meta_all["question_norm"] = meta_all["question"].astype(str).str.strip()
meta_all["answer_norm"] = meta_all["answer"].apply(basic_norm)

train_df = meta_all[meta_all["split"] == "train"].reset_index(drop=True)
answer_counts = train_df["answer_norm"].value_counts()
TOP_ANSWERS = answer_counts.head(TOP_K).index.tolist()
print("Top-K answers (train):", len(TOP_ANSWERS))

meta_all = meta_all[meta_all["split"].isin(SPLITS)].reset_index(drop=True)
print({s: len(meta_all[meta_all['split'] == s]) for s in SPLITS})
meta_all.head()


Top-K answers (train): 200
{'validation': 0, 'test': 15955}


,split,img_id,image_path,question,answer,question_type,answer_type,question_class,complexity,original,orig_height,orig_width,question_norm,answer_norm
0,test,clb0kvxwr92ng074yc6jndr8l,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,What type of polyp is observed in the gastroin...,no polypoid lesions identified,NaN,NaN,['polyp_type'],1,"[\n{\n""q"": ""What type of polyp is present?"",\n...",NaN,NaN,What type of polyp is observed in the gastroin...,no polypoid lesions identified
1,test,cl8k2u1q41ehr0832aze8a3c5,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,Have all identified polyps been successfully r...,residual polyps remain,NaN,NaN,['polyp_removal_status'],1,"[\n{\n""q"": ""Have all polyps been removed?"",\n""...",NaN,NaN,Have all identified polyps been successfully r...,residual polyps remain
2,test,clb0kvxvx91c4074y2e107c5l,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,Is there any textual content visible?,No visible text observed,NaN,NaN,['text_presence'],1,"[\n{\n""q"": ""Is there text?"",\n""a"": ""no""\n}\n]",NaN,NaN,Is there any textual content visible?,no visible text observed
3,test,clb0kvxuv8zrw074y9iwrgb8n,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,Is there any green or black box artefact prese...,"No green or black box artefact is present, z-l...",NaN,NaN,['box_artifact_presence' 'landmark_presence' '...,3,"[\n{\n""q"": ""Is there a green/black box artefac...",NaN,NaN,Is there any green or black box artefact prese...,no green or black box artefact is present z-li...
4,test,cla820gm5s5br071uapqa4zrj,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,"How many polyps are present, what colors are t...","No polyps are observed, multiple colors includ...",NaN,NaN,['polyp_count' 'abnormality_color' 'instrument...,3,"[\n{\n""q"": ""How many polyps are in the image?""...",NaN,NaN,"How many polyps are present, what colors are t...",no polyps are observed multiple colors includi...



### Normalization + mapping to top-K
- Lowercase and strip punctuation/extra spaces.
- Map predictions to the top-K answer list; if no good match, map to `other`.
- Ground-truth answers are also mapped to top-K (else `other`) so metrics stay comparable.


In [5]:

import difflib

_top_set = set(TOP_ANSWERS)


def normalize_answer(text: str) -> str:
    t = str(text).lower().strip()
    t = re.sub(r"[^a-z0-9\s\-]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def map_to_topk(ans_norm: str) -> str:
    if ans_norm in _top_set:
        return ans_norm
    match = difflib.get_close_matches(ans_norm, TOP_ANSWERS, n=1, cutoff=0.75)
    if match:
        return match[0]
    return "other"



### Load LLaVA-1.5 7B (quantized if possible)
Prefers 4-bit via bitsandbytes; falls back to 8-bit or full precision depending on availability.


In [6]:

quant_config = None
if torch.cuda.is_available() and BitsAndBytesConfig is not None:
    if USE_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
        print("Using 4-bit quantization (bitsandbytes).")
    elif USE_8BIT:
        quant_config = BitsAndBytesConfig(load_in_8bit=True)
        print("Using 8-bit quantization (bitsandbytes).")
elif USE_4BIT or USE_8BIT:
    print("Quantization requested but bitsandbytes is unavailable; using full precision.")
try:
    processor = AutoProcessor.from_pretrained(MODEL_NAME, use_fast=True)
except Exception as e:
    raise ImportError(
        "Failed to load processor. If you see a SentencePiece error, install it with: "
        "`pip install sentencepiece` (or `conda install -c conda-forge sentencepiece`)."
    ) from e
load_kwargs = {
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
    "low_cpu_mem_usage": True,
}
if torch.cuda.is_available():
    load_kwargs["device_map"] = "auto"
if quant_config is not None:
    load_kwargs["quantization_config"] = quant_config

model = LlavaForConditionalGeneration.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()

if hasattr(model, "hf_device_map"):
    device_vals = set(str(d) for d in model.hf_device_map.values())
    primary = next((d for d in device_vals if "cuda" in d), None)
    if primary is None:
        primary = next((d for d in device_vals if d not in {"cpu", "disk"}), "cpu")
    # Normalize device id like "0" to "cuda:0"
    if isinstance(primary, int):
        primary = f"cuda:{primary}"
    elif isinstance(primary, str) and primary.isdigit():
        primary = f"cuda:{primary}"
    PRIMARY_DEVICE = torch.device(primary)
else:
    PRIMARY_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Primary device for inputs:", PRIMARY_DEVICE)


Quantization requested but bitsandbytes is unavailable; using full precision.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Primary device for inputs: cuda:0



### Prompting + generation helpers
Uses the model chat template when available; otherwise falls back to `USER: <image> Question: ... ASSISTANT:` format.


In [7]:


def format_prompt(question: str) -> str:
    q = question.strip()
    if hasattr(processor, "apply_chat_template"):
        conversation = [
            {"role": "system", "content": "You are a concise medical VQA assistant."},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
        ]
        return processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    return f"USER: <image>\nQuestion: {q}\nASSISTANT:"


def postprocess_output(text: str) -> str:
    cleaned = text
    if "ASSISTANT:" in cleaned:
        cleaned = cleaned.split("ASSISTANT:")[-1]
    return cleaned.strip()


def chunk_indices(n: int, batch_size: int):
    for start in range(0, n, batch_size):
        yield start, min(start + batch_size, n)


def generate_for_split(df_split: pd.DataFrame, split_name: str) -> pd.DataFrame:
    df_split = df_split.reset_index(drop=True).copy()
    preds = []
    for start, end in tqdm(list(chunk_indices(len(df_split), BATCH_SIZE)), desc=f"{split_name} gens"):
        batch = df_split.iloc[start:end]
        images = [Image.open(p).convert("RGB") for p in batch["image_path"]]
        prompts = [format_prompt(q) for q in batch["question"]]
        inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(PRIMARY_DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        decoded = processor.batch_decode(out_ids, skip_special_tokens=True)
        preds.extend(postprocess_output(t) for t in decoded)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df_split["pred_raw"] = preds
    df_split["pred_norm"] = df_split["pred_raw"].apply(normalize_answer)
    df_split["pred_mapped"] = df_split["pred_norm"].apply(map_to_topk)
    df_split["gt_norm"] = df_split["answer_norm"].apply(normalize_answer)
    df_split["gt_mapped"] = df_split["gt_norm"].apply(map_to_topk)
    df_split["is_oov"] = df_split["pred_mapped"] == "other"
    return df_split


def compute_metrics(df: pd.DataFrame, split_name: str):
    labels_eval = TOP_ANSWERS + ["other"]
    metrics = {
        "split": split_name,
        "n": int(len(df)),
        "mapped_acc": float((df["pred_mapped"] == df["gt_mapped"]).mean()),
        "macro_f1": float(f1_score(df["gt_mapped"], df["pred_mapped"], labels=labels_eval, average="macro", zero_division=0)),
        "oov_rate": float(df["is_oov"].mean()),
    }
    report = classification_report(
        df["gt_mapped"], df["pred_mapped"], labels=labels_eval, output_dict=True, zero_division=0
    )
    per_class = pd.DataFrame(report).T
    return metrics, per_class



### Run zero-shot inference on validation + test
Saves per-sample predictions, per-class metrics, metrics summary, and a 20-row qualitative table per split.


In [8]:

all_metrics: List[Dict] = []
per_class_store: Dict[str, pd.DataFrame] = {}
qual_store: Dict[str, pd.DataFrame] = {}

for split in SPLITS:
    df_split = meta_all[meta_all["split"] == split].copy()
    if SAMPLE_N is not None:
        df_split = df_split.sample(min(SAMPLE_N, len(df_split)), random_state=SEED).reset_index(drop=True)
    print(f"=== {split.upper()} ({len(df_split)} samples) ===")
    df_pred = generate_for_split(df_split, split)
    metrics, per_class = compute_metrics(df_pred, split)
    all_metrics.append(metrics)
    per_class_store[split] = per_class

    df_pred.to_csv(OUT_DIR / f"predictions_{split}.csv", index=False)
    per_class.to_csv(OUT_DIR / f"per_class_{split}.csv")

    qual = df_pred.sample(n=min(20, len(df_pred)), random_state=SEED)[
        ["img_id", "question", "pred_raw", "pred_mapped", "gt_mapped", "answer", "is_oov"]
    ]
    qual.to_csv(OUT_DIR / f"qualitative_{split}.csv", index=False)
    qual_store[split] = qual

metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(OUT_DIR / "metrics_summary.csv", index=False)
with open(OUT_DIR / "metrics_summary.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

metrics_df


=== VALIDATION (0 samples) ===


validation gens: 0it [00:00, ?it/s]

=== TEST (15955 samples) ===


test gens:   0%|          | 0/7978 [00:00<?, ?it/s]

,split,n,mapped_acc,macro_f1,oov_rate
0,validation,0,NaN,0.000000,NaN
1,test,15955,0.561642,0.005817,0.973049



### Quick qualitative peek (test split)


In [9]:
qual_store.get("test", pd.DataFrame()).head(20)

,img_id,question,pred_raw,pred_mapped,gt_mapped,answer,is_oov
3780,cla820gmds5lb071ug0oedfn0,Are there any instruments visible in the image...,"Yes, there are instruments visible in the imag...",other,no instruments are visible in the image,No instruments are visible and text is present,True
4925,clb0lbwzadoyg086u671x4ucx,Are there any instruments present in the image...,"Yes, there is a medical instrument present in ...",other,other,A single tube is visible in the field of view ...,True
15504,clb0kvxwj929c074yhbmvg5fy,What abnormalities are visible in the image?,"In the image, there are two visible abnormalit...",other,other,evidence of Barrett's esophagus and esophagitis,True
88,cla820gmcs5kb071uh99a6wnr,"Have all polyps been removed, how many finding...","In the image, the anatomical landmark is the c...",other,other,"No polyps remain, one abnormal finding is pres...",True
15754,cla820gm7s5e7071u45cwfvua,Are there any abnormalities or anatomical land...,"Yes, there is a large, red, and inflamed area ...",other,no identifiable anatomical landmarks observed,No significant abnormalities or identifiable a...,True
2742,cla820gn5s6kr071ue6jighxj,"How many polyps are present, what procedure wa...",The image shows a large number of polyps in th...,other,other,"No polyps are visualized, the image is from a ...",True
11020,clb0kvxvj90so074yb5zic15r,"What abnormalities, instruments, and procedure...","In the image, there is a close-up view of a pe...",other,other,Evidence of oesophagitis is present with no in...,True
15092,clb0lbx1jdqig086u8y07fl95,"Are there any instruments visible, how many fi...","Yes, there is a black instrument visible in th...",other,other,A single finding is present with a tube visibl...,True
9222,clb0lbx08dq7c086u4mcm6bqm,"Is there any text visible, are all polyps remo...","Yes, there is text visible in the image, which...",other,other,"No text is visible, no polyps are noted, and o...",True
13287,clb0kvxvf90lw074y2dav4gy7,Are there any text elements present and is the...,"Yes, there is a text element present in the im...",other,no text observed and no polyps identified,No text observed and no polyp detected,True
